In [1]:
# RELOAD KERNEL IF CHANGES MADE TO quadrotor.py, quad_sim.py, create_animation.py, etc
import numpy as np
from math import sin, cos, pi
from trajectories import *
from scipy.integrate import solve_ivp
import matplotlib.pyplot as plt
import importlib
from numpy.linalg import inv

from quad_sim import simulate_quadrotor

# Need to reload the module to use the latest code
import quadrotor
importlib.reload(quadrotor)
from quadrotor import Quadrotor

In [2]:
# quadrotor controller input params:
# Weights of LQR cost
R = np.eye(2);
Q = np.diag([10, 10, 1, 1, 1, 1]);
Qf = Q;

# Quadrotor internal params
m = 1
a = 0.25
I = 0.0625

# End time of the simulation
tf = 2*pi;

In [3]:
# Defining necessary ODEs for Ldot and linearized dynamics A(t), B(t) for experimentation
# Linearized Dynamics of _f(x, u) in quad_sim to get xdot \approx A(t)x_e + B(t)x_e
def A(t):
    x, u = x_d(t), u_d(t)
    theta, u1, u2 = x[2], u[0], u[1]

    dq_ddotdq = np.array(
        [
            [0, 0, (-np.cos(theta) / m) * (u1 + u2)],
            [0, 0, (-np.sin(theta) / m) * (u1 + u2)],
            [0, 0, 0],
        ]
    )
    A = np.block([[np.zeros((3, 3)), np.eye(3)], [dq_ddotdq, np.zeros((3, 3))]])
    return A

def B(t):
    x, u = x_d(t), u_d(t)
    theta = x[2]
    M = np.array(
        [
            [-np.sin(theta) / m, -np.sin(theta) / m],
            [np.cos(theta) / m, np.cos(theta) / m],
            [a / I, -a / I],
        ]
    )
    B = np.block([[np.zeros((3, 2))], [M]])
    return B

def Ldot(t, L, Q=Q, R=R):
    L = L.reshape((6,6))
    A_t = A(t)
    B_t = B(t)

    dLdt = np.zeros((6, 6))
    dLdt -= 0.5 * Q @ inv(L).transpose()
    dLdt -= A_t.transpose() @ L
    dLdt += 0.5 * L @ L.transpose() @ B_t @ inv(R) @ B_t.transpose() @ L

    return dLdt.reshape(-1)

In [4]:
# Computing L(tf) via cholesky and then integrating backwards in time:

# Get L(tf) L(tf).T = S(tf) by decomposing S(tf) using Cholesky decomposition
Lf = np.linalg.cholesky(Qf) # returns lower by default, no need to transpose
# print(Lf) # diagonal

# We need to reshape L0 from a square matrix into a row vector to pass into solve_ivp()
lf = np.reshape(Lf, (36))

# L must be integrated backwards, solve_ivp handles for us if we pass it inverted tspan from tf-> #0
tspan = [tf, 0]  # noqa: F841
sol = solve_ivp(Ldot, tspan, lf, dense_output=True)
t = sol.t
l = sol.y
t[:5], t[-1] # time decreasing from tf -> 0

(array([6.28318531, 6.2829676 , 6.28079049, 6.25901944, 6.23096945]),
 np.float64(0.0))

In [5]:
# We don't need to 
l_spline = sol.sol

Lfn = l_spline(tf).reshape((6, 6))
Lfn @ Lfn

array([[10.,  0.,  0.,  0.,  0.,  0.],
       [ 0., 10.,  0.,  0.,  0.,  0.],
       [ 0.,  0.,  1.,  0.,  0.,  0.],
       [ 0.,  0.,  0.,  1.,  0.,  0.],
       [ 0.,  0.,  0.,  0.,  1.,  0.],
       [ 0.,  0.,  0.,  0.,  0.,  1.]])

In [13]:
"""
Simulate quadrotor
"""
# Construct our quadrotor controller 
quadrotor = Quadrotor(Q, R, Qf, tf);

# Set quadrotor's initial state and simulate
x0 = x_d(0.0) + np.random.normal(size=(6,))*4
x, u, t = simulate_quadrotor(x0, tf, quadrotor)

In [14]:
%matplotlib inline
"""
Create animation 
"""
import create_animation
importlib.reload(create_animation)
from create_animation import create_animation, save_trajectory_animation

# Number of poses to visualize 
# TODO: set this value to 60 for the final plots
n_frame = 30

anim, fig = create_animation(x, x_d, tf, n_frame)
plt.close()

anim

In [8]:
# Set to True to save this animation (+ a log of x0 and the trajectory error
# x(t) - x_d(t) at every timestep) under quadrotor/trajectory-anims/
SAVE_VIDEO = True
if SAVE_VIDEO:
    save_trajectory_animation(anim, x, t, x_d, x0, filename="converging-trajectory")

x0 = [3.9545811334616383, 8.181641224927334, -2.228477314264832, 3.8197960368720016, -3.313070158450963, -7.64819475024716]
t=0.0000  x_e=[ 0.95458113  8.18164122 -2.52525619  3.81979604 -6.31307016 -7.73371673]
t=0.0010  x_e=[ 0.95840243  8.17532816 -2.53298979  3.82248054 -6.32263497 -7.39380921]
t=0.0020  x_e=[ 0.96222641  8.16900552 -2.54038348  3.82521984 -6.33223591 -7.05763555]
t=0.0030  x_e=[ 0.96605313  8.16267329 -2.547441    3.8280134  -6.34187397 -6.72516927]
t=0.0040  x_e=[ 0.96988264  8.15633142 -2.55416605  3.83086073 -6.35155013 -6.39638395]
t=0.0050  x_e=[ 0.973715    8.14997988 -2.56056232  3.83376135 -6.36126525 -6.07125333]
t=0.0060  x_e=[ 0.97755027  8.14361862 -2.56663346  3.83671483 -6.3710202  -5.74975125]
t=0.0070  x_e=[ 0.98138848  8.13724761 -2.57238309  3.83972077 -6.38081574 -5.43185169]
t=0.0080  x_e=[ 0.9852297   8.1308668  -2.57781483  3.84277882 -6.39065263 -5.11752872]
t=0.0090  x_e=[ 0.98907398  8.12447616 -2.58293224  3.84588864 -6.40053154 -4.806756

In [9]:
np.linalg.norm(x0 - x_d(0))

np.float64(13.728928151572415)

In [10]:
x0, x_d(0)

(array([ 3.95458113,  8.18164122, -2.22847731,  3.81979604, -3.31307016,
        -7.64819475]),
 array([ 3.        ,  0.        ,  0.29677887, -0.        ,  3.        ,
         0.08552198]))